## Begin with the story

Imagine someone says, "I have five apples and my friend has ten. How many do we have together?" You choose addition and get 15 apples. Next, someone asks how far away a tree is, given a height 10 metres above eye level and an elevation angle of 30 degrees on level ground. This needs trigonometry: distance = height / tan(angle), or about 17.32 metres. If the angle is missing, you must ask for it.

**Your project is an Intelligent Calculator.** You will build two small Python functions, then give an existing language model access to them. From an English question, the model should request the appropriate function with the right values. Python calculates the result, which returns to the model for a reply. You will inspect the tool request, inputs, returned result and final answer to check what really happened.

LangChain connects the model and tools and assembles the agent using `create_agent`. Its built-in LangGraph runtime carries out the model-and-tool loop. You do not write a custom graph in this Intro Build. LangSmith records execution for inspection in the instructor notebook. The instructor's MCP demo exposes the calculator to an external client.

First connect to the model and observe a plain reply. Then build and directly test the two calculators. Next assemble the agent and inspect the apples, tree and missing-angle cases. Work individually, predict before running, and explain every checkpoint. The manual tells the story and explains each stage before its code.

---

# Intro Build · Instructor Complete
Run the same 12 cells using your own Groq key. The extra cells below capture LangSmith traces. Save your rehearsed outputs before class. If campus Wi-Fi fails, connect your laptop to your phone hotspot and project this notebook.

### Cell 1 · Install the notebook packages

In [ ]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

### Cell 2 · Connect your model client

In [ ]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ").strip()
if not os.environ["GROQ_API_KEY"]:
    raise ValueError("Enter a Groq API key in the hidden prompt.")
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0, timeout=30, max_retries=1)
print("Model client configured. Cell 3 tests the connection.")

### Cell 3 · Ask the plain model

In [ ]:
APPLES = "One person brings five apples and another brings ten. They put them together. How many apples are there?"
plain_reply = model.invoke([("human", APPLES)])
print(plain_reply.content)

### Cell 4 · Send a system instruction

In [ ]:
role_reply = model.invoke([
    ("system", "You are a patient math tutor. Give one short sentence with the answer and its units."),
    ("human", APPLES),
])
print(role_reply.content)

### Cell 5 · Define the addition tool

In [ ]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.name)

### Cell 6 · Define the tree-distance tool

In [ ]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.name)

### Cell 7 · Test both tools directly

In [ ]:
print("Apples:", add.invoke({"a": 5, "b": 10}))
print("Tree distance in metres:", tree_distance.invoke({"height_m": 10, "angle_deg": 30}))
print("Invalid angle:", tree_distance.invoke({"height_m": 10, "angle_deg": 0}))

### Cell 8 · Assemble the LangChain agent

In [ ]:
from langchain.agents import create_agent

INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)
print("Agent ready")

### Cell 9 · Add a tool-message viewer

In [ ]:
from langchain_core.messages import ToolMessage

def solve(problem, *, run_name="intro_calculator"):
    return agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": run_name},
    )

def show_run(problem):
    result = solve(problem)
    call_count = 0
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            call_count += 1
            print("Tool requested:", call["name"], call["args"])
        if isinstance(message, ToolMessage):
            print("Tool returned:", message.name, message.content)
    print("Tool requests:", call_count)
    print("Final answer:", result["messages"][-1].content)
    return result

print("Viewer ready")

### Cell 10 · Run the apples problem

In [ ]:
apples_run = show_run(APPLES)

### Cell 11 · Run the complete tree problem

In [ ]:
TREE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "The angle of elevation to the top is 30 degrees. "
    "How far away is the tree horizontally?"
)
tree_run = show_run(TREE)

### Cell 12 · Run the tree problem without the angle

In [ ]:
MISSING_ANGLE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "How far away is the tree horizontally?"
)
missing_run = show_run(MISSING_ANGLE)

## Instructor extension · Configure LangSmith
Choose the destination that exists in your LangSmith account. Agent-based workspaces use an agent ID and environment. Project-based workspaces use a project name. Enter the API URL shown by your workspace if it uses a region outside the default US endpoint. Keep credentials out of screen recordings.

In [ ]:
from langsmith import Client, tracing_context

destination = input("LangSmith destination type (agent/project): ").strip().lower()
if destination == "agent":
    os.environ.pop("LANGSMITH_PROJECT", None)
    os.environ["LANGSMITH_AGENT_ID"] = input("Agent ID: ").strip()
    os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = input(
        "Environment (for example local): "
    ).strip()
elif destination == "project":
    os.environ.pop("LANGSMITH_AGENT_ID", None)
    os.environ.pop("LANGSMITH_AGENT_ENVIRONMENT", None)
    os.environ["LANGSMITH_PROJECT"] = input("Project name: ").strip()
else:
    raise ValueError("Enter agent or project, then rerun this cell.")

endpoint = input("LangSmith API URL (blank for default US): ").strip()
if endpoint:
    os.environ["LANGSMITH_ENDPOINT"] = endpoint
workspace_id = input("Workspace ID (blank if unnecessary): ").strip()
if workspace_id:
    os.environ["LANGSMITH_WORKSPACE_ID"] = workspace_id
smith_key = getpass("LangSmith API key: ").strip()
smith_client = Client(api_key=smith_key)
print("Tracing client configured.")


## Instructor extension · Send the three traces
Run this after the 12 student cells. Open LangSmith and identify the three named runs. Confirm the missing-angle run actually made zero calculator calls.

In [ ]:
with tracing_context(enabled=True, client=smith_client):
    solve(APPLES, run_name="intro_apples")
    solve(TREE, run_name="intro_tree")
    solve(MISSING_ANGLE, run_name="intro_missing_angle")
smith_client.flush()
print("Look for intro_apples, intro_tree and intro_missing_angle.")


## Instructor extension · Record the environment
After the live rehearsal succeeds, run the cell below and save this notebook with its outputs. This records the versions you actually used.

In [ ]:
from importlib.metadata import version
for package in ("langchain", "langgraph", "langchain-groq", "langsmith"):
    print(package, version(package))
